In [2]:
import pandas as pd
from pathlib import Path

# ===== 변경된 경로 =====
p2022 = Path(r"C:/8ssible-Healing-Seoul-Analysis/branch_SG/8SSIBLE_SG_DATA/S-DoT_NATURE_2022년(2022.01.03~2023.01.01)")
p2023 = Path(r"C:/8ssible-Healing-Seoul-Analysis/branch_SG/8SSIBLE_SG_DATA/S-DoT_NATURE_2023년(2023.01.01~2024.01.01)")
p2024 = Path(r"C:/8ssible-Healing-Seoul-Analysis/branch_SG/8SSIBLE_SG_DATA/S-DoT_NATURE_2024년(2024.01.01~2024.12.31)")
roots = [p2022, p2023, p2024]

# 컬럼명
COL_DT = "측정시간"
COL_GU = "자치구"
COL_T  = "온도 평균(℃)"
COL_H  = "습도 평균(%)"

required = [COL_DT, COL_GU, COL_T, COL_H]

agg_parts = []

for root in roots:
    for f in root.rglob("*.csv"):
        header = None
        enc_used = None

        for enc in ["utf-8-sig", "cp949", "euc-kr"]:
            try:
                header = pd.read_csv(f, nrows=0, encoding=enc)
                enc_used = enc
                break
            except Exception:
                pass

        if header is None or not set(required).issubset(set(header.columns)):
            continue

        for chunk in pd.read_csv(f, usecols=required, encoding=enc_used, chunksize=200000, low_memory=False):
            dt = pd.to_datetime(chunk[COL_DT].astype(str).str.replace("_", " ", regex=False), errors="coerce")
            t  = pd.to_numeric(chunk[COL_T], errors="coerce")
            h  = pd.to_numeric(chunk[COL_H], errors="coerce")
            gu = chunk[COL_GU].astype(str).str.strip()

            valid = dt.notna() & t.notna() & h.notna() & gu.ne("")
            if not valid.any():
                continue

            dt, t, h, gu = dt[valid], t[valid], h[valid], gu[valid]

            # DI = 1.8T - 0.55(1-H)(1.8T-26) + 32, H는 0~1
            h_ratio = h / 100.0
            di = 1.8 * t - 0.55 * (1 - h_ratio) * (1.8 * t - 26) + 32

            temp_df = pd.DataFrame({
                "년도": dt.dt.year,
                "월": dt.dt.month,
                "자치구": gu,
                "온도합": t,
                "습도합": h,
                "불쾌지수합": di,
                "건수": 1,
            })

            agg_parts.append(temp_df.groupby(["년도", "월", "자치구"], as_index=False).sum(numeric_only=True))

final = (
    pd.concat(agg_parts, ignore_index=True)
      .groupby(["년도", "월", "자치구"], as_index=False)
      .sum(numeric_only=True)
)

final["온도 평균(℃)"] = final["온도합"] / final["건수"]
final["습도 평균(%)"] = final["습도합"] / final["건수"]
final["파생컬럼(불쾌지수)"] = final["불쾌지수합"] / final["건수"]

result_df = final[["년도", "월", "자치구", "온도 평균(℃)", "습도 평균(%)", "파생컬럼(불쾌지수)"]]
result_df = result_df.sort_values(["년도", "월", "자치구"]).reset_index(drop=True)
result_df[["온도 평균(℃)", "습도 평균(%)", "파생컬럼(불쾌지수)"]] = \
    result_df[["온도 평균(℃)", "습도 평균(%)", "파생컬럼(불쾌지수)"]].round(2)

display(result_df)


,년도,월,자치구,온도 평균(℃),습도 평균(%),파생컬럼(불쾌지수)
0,2022,12,Dobong-gu,0.09,61.91,37.45
1,2022,12,Dongdaemun-gu,0.97,54.81,39.76
2,2022,12,Dongjak-gu,1.38,56.87,40.04
3,2022,12,Eunpyeong-gu,0.26,58.41,38.26
4,2022,12,Gangbuk-gu,-3.25,59.27,33.28
...,...,...,...,...,...,...
652,2024,12,Seoul_Grand_Park,1.19,56.32,39.74
653,2024,12,Songpa-gu,2.44,52.43,42.09
654,2024,12,Yangcheon-gu,2.42,54.69,41.82
655,2024,12,Yeongdeungpo-gu,2.71,52.96,42.45
